# Mars band source comparison

## tl;dr

The same PNG with the gray band is present in Mars-Bench multi. The older THEMIS v2 source has a black band: 22,500 source zeros become 126 in Mars-Bench; all 239,644 other pixels are identical. A newer THEMIS v11 crop shows the same terrain without the band, supported by 820/823 geometric matches. No dataset images or reference answers are changed.

## Context & Methods

Exact source URLs, SHA-256, ZIP member offsets/CRC and geographic crop coordinates are in data_orbital/source_comparison_v1/comparison.json. Only one PNG plus the ZIP index and a small modern-mosaic crop were retrieved, not full datasets. One older 38 MB parent tile was downloaded with resume.

### Key Assumptions

The binary/multi crater sets are related. Newer mosaic resolution and contributing acquisitions differ; this is not a controlled corruption counterfactual. The older page mentions transparency but the actual file is mode L without alpha. Zero counts alone do not establish completeness; scene registration and visual inspection are also used. GT transfer and replacement need a separate task.

Sources: [Mars-Bench multi](https://zenodo.org/records/15494713), [THEMIS v2](https://www.mars.asu.edu/data/thm_dir/), [THEMIS v11](https://www.mars.asu.edu/data/thm_dir_100m/).

## Data

### 1. Verify cached identities

In [1]:
from pathlib import Path
import json,hashlib,sys
import numpy as np,cv2
from PIL import Image
WORKSPACE=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src/planetary_vlm').is_dir() and (p/'data_orbital').is_dir())
FOLDER=WORKSPACE/'data_orbital/source_comparison_v1'
record=json.loads((FOLDER/'comparison.json').read_text(encoding='utf-8'))
def sha(path):
    with path.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
original=WORKSPACE/record['local_original']['path']
assert sha(original)==record['local_original']['sha256']
for name,expected in record['evidence_png_sha256'].items():assert sha(FOLDER/name)==expected
assert not record['marsbench_multi']['archive_full_downloaded'] and not record['themis_v11']['full_mosaic_downloaded']
print(json.dumps({'source_id':record['source_image_id'],'verified_files':len(record['evidence_png_sha256']),'zip_bytes_read':record['marsbench_multi']['http_body_bytes_for_selected_png_and_index'],'modern_crop_bytes_read':record['themis_v11']['http_body_bytes_for_crop']},indent=2))

{
  "source_id": "thm_dir_N-30_330_179.png",
  "verified_files": 6,
  "zip_bytes_read": 1891964,
  "modern_crop_bytes_read": 1539680
}


## Results

### 2. Second dataset and older parent

In [2]:
assert original.read_bytes()==(FOLDER/'marsbench_multi_zenodo.png').read_bytes()
old=np.asarray(Image.open(original))
with Image.open(FOLDER/'themis_v2_parent.png') as tile:
    assert tile.mode=='L' and 'transparency' not in tile.info
    source=np.asarray(tile.crop(tuple(record['themis_v2']['crop_xyxy'])))
assert np.array_equal(source,np.asarray(Image.open(FOLDER/'themis_v2_parent_crop.png')))
gap=source==0
assert int(gap.sum())==22500 and int(np.count_nonzero(old!=source))==22500
assert np.array_equal(old[~gap],source[~gap]) and np.all(old[gap]==126)
print(json.dumps({'multi_PNG_identical':True,'source_band_value':0,'MarsBench_band_value':126,'band_pixels':22500,'other_pixels_identical':int((~gap).sum()),'band_percent':100*float(gap.mean())},indent=2))

{
  "multi_PNG_identical": true,
  "source_band_value": 0,
  "MarsBench_band_value": 126,
  "band_pixels": 22500,
  "other_pixels_identical": 239644,
  "band_percent": 8.58306884765625
}


The source band predates our assembly. Its gray rendering is present in the published second crater dataset, not introduced by CVAT or our export.

### 3. Newer mosaic scene correspondence

In [3]:
native=Image.open(FOLDER/'themis_v11_geographic_crop.png')
assert native.size==(1185,1186)
new=np.asarray(native.resize((512,512),Image.Resampling.BOX))
assert np.array_equal(new,np.asarray(Image.open(FOLDER/'themis_v11_geographic_crop_display_512.png')))
cv2.setRNGSeed(0);sift=cv2.SIFT_create(nfeatures=3000)
ka,da=sift.detectAndCompute(old,None);kb,db=sift.detectAndCompute(new,None)
good=[m for m,n in cv2.BFMatcher().knnMatch(da,db,k=2) if m.distance<.7*n.distance]
a=np.float32([ka[m.queryIdx].pt for m in good]);b=np.float32([kb[m.trainIdx].pt for m in good])
M,mask=cv2.estimateAffinePartial2D(a,b,method=cv2.RANSAC,ransacReprojThreshold=3)
assert len(good)==823 and int(mask.sum())==820
displacement=float(np.median(np.linalg.norm(a[mask.ravel()>0]-b[mask.ravel()>0],axis=1)))
assert displacement<2
print(json.dumps({'matches':len(good),'inliers':int(mask.sum()),'median_display_displacement_px':displacement,'affine':M.tolist(),'new_crop_zero_pixels':int(np.count_nonzero(np.asarray(native)==0))},indent=2))

{
  "matches": 823,
  "inliers": 820,
  "median_display_displacement_px": 1.253058910369873,
  "affine": [
    [
      0.999916509419274,
      -0.00027064115450851434,
      -0.9805877122647488
    ],
    [
      0.00027064115450851434,
      0.999916509419274,
      -0.7344853402959707
    ]
  ],
  "new_crop_zero_pixels": 0
}


Visual evidence: the native source band is visible in the old crop, while actual surface is visible in the newer crop.

![Old source](../data_orbital/source_comparison_v1/themis_v2_parent_crop.png)

![Newer mosaic](../data_orbital/source_comparison_v1/themis_v11_geographic_crop_display_512.png)

### 4. Preserve current dataset and unknowns

In [4]:
sys.path.insert(0,str(WORKSPACE/'src'))
from planetary_vlm.datasets.crater_annotations import verify_crater_build
from planetary_vlm.io import read_jsonl
verification=verify_crater_build(WORKSPACE,WORKSPACE/'data_orbital/crater_annotations_v4')
assert verification['integrity_verified'] and verification['reference_ready_questions']==0
unknown=[q for q in read_jsonl(WORKSPACE/'data_orbital/crater_annotations_v4/questions.jsonl') if q['reference_status']=='UNKNOWN_REFERENCE']
assert len(unknown)==3 and {q['sample_id'] for q in unknown}=={'mars_seg_018'}
print(json.dumps(verification,indent=2))

{
  "schema": "crater-annotations-v1",
  "integrity_verified": true,
  "bundle_sha256": "2c81ac47fcba3ba1d8867f8f2e1c6fb9d153a220b71c4aa9fa98099dfe44d8a3",
  "images": 100,
  "candidate_questions": 400,
  "reference_ready_questions": 0,
  "ready_for_scored_inference": false,
  "scientific_result": false
}


## Takeaways

The band is present in the second crater dataset and older source mosaic, but absent on visual inspection of the registered newer-mosaic crop. Use this as separate evidence, not an automatic image replacement or accepted GT. The original 397 candidate answers and 3 unknowns remain unchanged.

Acquisition reproducibility: comparison.json records exact archive/member offsets, CRC, byte counts and hashes. The older crop is [7168,5632,7680,6144]. The newer P5 PGM has a 19-byte header and 35565-pixel rows; request bytes 19+y*35565+34380 through 19+y*35565+35564 for y in [13040,14226), at most 100 ranges per request, validating HTTP 206 and each multipart Content-Range against total size 632416849. Concatenate into an 1185 by 1186 uint8 image. The 512-square image is only a BOX-resized display.

Validation: nbformat checked and four ordinary Python cells executed sequentially with captured stdout, not via a Jupyter kernel. Optional kernel validation with nbclient/ipykernel/nbconvert installed: python -m jupyter nbconvert --execute --to notebook --inplace notebooks/mars_band_source_comparison.ipynb. No new packages were installed.